# Notebook 14 — Map H2 Pipeline Cost Models to Geospatial Schema Tables -- DEPRECATED

## Purpose

This notebook translates the selected hydrogen pipeline cost models produced by the preceding cost-model workflow into structures compatible with the Geospatial-CANOE/TEMOA schema.

The notebook separates the workflow into two layers:

1. **Topology-free cost templates**, which preserve the selected cost-model parameters independently of any particular graph.
2. **Development-graph mappings**, which apply edge distances and region identifiers to confirm how the templates will populate the final schema tables.

The graph-specific mappings in this notebook are validation products. The canonical outputs exported for later use by `build_schema.py` remain topology-free.

---

## Cost-table mapping

The selected H2 pipeline cost models are mapped as follows:

| Cost model | Representation | TEMOA table |
|---|---|---|
| Pipeline CAPEX | Piecewise-linear capacity and cost bounds | `ETLSegment` |
| Fixed OPEX | Linear distance-based cost model | `CostFixed` |
| Variable OPEX | Linear distance-based cost model | `CostVariable` |

Pipeline CAPEX is represented entirely through `ETLSegment`; no separate H2 pipeline rows are constructed for `CostInvest`.

Fixed and variable OPEX are treated as distinct cost types under the broader pipeline OPEX category:

- `fixed_opex` produces `CostFixed` rows.
- `variable_opex` produces `CostVariable` rows.

---

## Inherited geospatial cost convention

For fixed and variable OPEX, edge-level costs follow the transport-cost mapping inherited from the earlier geospatial workflow:

\[
\text{cost}_{ij}
=
\text{intercept cost}
+
\text{coefficient per km}\times d_{ij}
\]

where \(d_{ij}\) is the distance of the directed graph edge connecting regions \(i\) and \(j\).

The regression coefficients remain topology-free until they are combined with the selected development graph.

---

## Workflow

This notebook:

1. Loads the selected H2 pipeline cost models.
2. Extracts the selected linear fixed- and variable-OPEX models.
3. Constructs a topology-free four-segment CAPEX template.
4. Loads a processed development graph.
5. Maps the CAPEX template onto all graph edges for validation.
6. Constructs topology-free fixed- and variable-OPEX coefficient rows.
7. Maps both OPEX models onto all graph edges.
8. Validates every mapped CAPEX and OPEX row.
9. Exports the topology-free cost templates for integration into `build_schema.py`.

---

## Canonical outputs

The notebook exports:

- `h2_pipeline_etlsegment_template.csv`
- `h2_pipeline_opex_coefficients.csv`

These files contain no graph-region identifiers or edge distances.

`build_schema.py` will later:

- load the selected graph;
- apply edge distances;
- construct final `ETLSegment`, `CostFixed`, and `CostVariable` rows;
- combine them with the remaining model tables;
- write the completed TEMOA SQLite schema.

The graph-mapped DataFrames created here are retained as development and validation objects rather than canonical input files.

In [1]:
# =============================================================================
# Imports and project discovery
# =============================================================================

from pathlib import Path
import sys

import numpy as np
import pandas as pd

DATA_ID = "GEO001"

print(f"Active dataset ID: {DATA_ID}")


def find_project_root() -> Path:
    """Locate the Geospatial-CANOE repository root.

    Searches upward from the current working directory for a parent directory
    containing the project ``data_files`` and ``scripts`` directories.

    Returns
    -------
    Path
        Absolute path to the repository root.

    Raises
    ------
    FileNotFoundError
        If the repository root cannot be located.
    """

    for candidate in [
        Path.cwd().resolve(),
        *Path.cwd().resolve().parents,
    ]:
        if (
            (candidate / "data_files").is_dir()
            and (candidate / "scripts").is_dir()
        ):
            return candidate

    raise FileNotFoundError(
        "Could not locate the Geospatial-CANOE project root. "
        "Expected a parent directory containing both "
        "'data_files/' and 'scripts/'."
    )


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

Active dataset ID: GEO001
Project root: C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace


In [2]:
# =============================================================================
# Input and output paths
# =============================================================================

DATA_FILES = PROJECT_ROOT / "data_files"

H2_PIPELINE_COST_DIR = (
    DATA_FILES
    / "processed"
    / "costs"
    / "transport"
    / "pipelines"
    / "h2_pipeline"
)

INPUT_SELECTED_COST_MODEL_PATH = (
    H2_PIPELINE_COST_DIR
    / "h2_pipeline_cost_model_selection.csv"
)

PROCESSED_GRAPH_DIR = (
    DATA_FILES
    / "processed"
    / "graph"
)

GRAPH_EDGE_PATTERN = "*_graph_edges.csv"

MAPPED_PIPELINE_COST_DIR = (
    H2_PIPELINE_COST_DIR
    / "mapped"
)

print("Pipeline cost-mapping paths:")
print(
    "  Selected cost models: "
    f"{INPUT_SELECTED_COST_MODEL_PATH}"
)
print(
    "  Processed graph folder: "
    f"{PROCESSED_GRAPH_DIR}"
)
print(
    "  Graph-edge pattern: "
    f"{GRAPH_EDGE_PATTERN}"
)
print(
    "  Mapped-cost folder: "
    f"{MAPPED_PIPELINE_COST_DIR}"
)

Pipeline cost-mapping paths:
  Selected cost models: C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\processed\costs\transport\pipelines\h2_pipeline\h2_pipeline_cost_model_selection.csv
  Processed graph folder: C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\processed\graph
  Graph-edge pattern: *_graph_edges.csv
  Mapped-cost folder: C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\processed\costs\transport\pipelines\h2_pipeline\mapped


In [3]:
# =============================================================================
# Load and validate H2 pipeline cost inputs
# =============================================================================

def load_selected_cost_models(
    cost_model_path: Path,
) -> pd.DataFrame:
    """Load and validate the selected H2 pipeline regression models.

    Parameters
    ----------
    cost_model_path : Path
        Path to the selected H2 pipeline cost-model table produced by
        Notebook 13.

    Returns
    -------
    pd.DataFrame
        Validated selected cost models containing one CAPEX power regression,
        one fixed-OPEX linear regression, and one variable-OPEX linear
        regression.

    Raises
    ------
    FileNotFoundError
        If the selected cost-model table does not exist.
    ValueError
        If the table is empty, missing required columns, contains unexpected
        cost types or model types, contains invalid regression parameters, or
        includes more than one technology or commodity.
    """

    if not cost_model_path.exists():
        raise FileNotFoundError(
            "Selected H2 pipeline cost-model table not found: "
            f"{cost_model_path}"
        )

    selected_models = pd.read_csv(
        cost_model_path,
        encoding="utf-8",
    )

    if selected_models.empty:
        raise ValueError(
            "Selected H2 pipeline cost-model table is empty."
        )

    required_columns = [
        "technology",
        "commodity",
        "cost_type",
        "model_type",
        "slope",
        "intercept",
        "coefficient",
        "exponent",
        "capacity_min",
        "capacity_max",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in selected_models.columns
    ]

    if missing_columns:
        raise ValueError(
            "Selected cost-model table is missing columns: "
            f"{missing_columns}"
        )

    selected_models = selected_models.copy()

    string_columns = [
        "technology",
        "commodity",
        "cost_type",
        "model_type",
    ]

    for column in string_columns:
        selected_models[column] = (
            selected_models[column]
            .astype("string")
            .str.strip()
        )

        if selected_models[column].isna().any():
            raise ValueError(
                f"Selected cost-model column '{column}' contains "
                "missing values."
            )

        if (selected_models[column] == "").any():
            raise ValueError(
                f"Selected cost-model column '{column}' contains "
                "blank values."
            )

    expected_models = {
        "capex": "power",
        "fixed_opex": "linear",
        "variable_opex": "linear",
    }

    if selected_models["cost_type"].duplicated().any():
        duplicated_cost_types = (
            selected_models.loc[
                selected_models["cost_type"].duplicated(keep=False),
                "cost_type",
            ]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(
            "Selected cost-model table contains duplicate cost types: "
            f"{duplicated_cost_types}"
        )

    actual_cost_types = set(
        selected_models["cost_type"]
    )

    expected_cost_types = set(
        expected_models
    )

    if actual_cost_types != expected_cost_types:
        raise ValueError(
            "Selected cost-model table must contain exactly "
            f"{sorted(expected_cost_types)}, but contains "
            f"{sorted(actual_cost_types)}."
        )

    for cost_type, expected_model_type in expected_models.items():
        actual_model_type = selected_models.loc[
            selected_models["cost_type"] == cost_type,
            "model_type",
        ].iloc[0]

        if actual_model_type != expected_model_type:
            raise ValueError(
                f"Expected '{cost_type}' to use "
                f"'{expected_model_type}', but found "
                f"'{actual_model_type}'."
            )

    if selected_models["technology"].nunique() != 1:
        raise ValueError(
            "Selected cost models must contain exactly one technology."
        )

    if selected_models["commodity"].nunique() != 1:
        raise ValueError(
            "Selected cost models must contain exactly one commodity."
        )

    for column in [
        "capacity_min",
        "capacity_max",
    ]:
        selected_models[column] = pd.to_numeric(
            selected_models[column],
            errors="coerce",
        )

        if selected_models[column].isna().any():
            raise ValueError(
                f"Selected cost-model column '{column}' contains "
                "invalid values."
            )

        if not np.isfinite(selected_models[column]).all():
            raise ValueError(
                f"Selected cost-model column '{column}' contains "
                "non-finite values."
            )

        if (selected_models[column] <= 0).any():
            raise ValueError(
                f"Selected cost-model column '{column}' must be "
                "strictly positive."
            )

    if not (
        selected_models["capacity_max"]
        > selected_models["capacity_min"]
    ).all():
        raise ValueError(
            "Every selected cost model must have capacity_max greater "
            "than capacity_min."
        )

    if not np.allclose(
        selected_models["capacity_min"],
        selected_models["capacity_min"].iloc[0],
    ):
        raise ValueError(
            "Selected cost models do not share a common capacity_min."
        )

    if not np.allclose(
        selected_models["capacity_max"],
        selected_models["capacity_max"].iloc[0],
    ):
        raise ValueError(
            "Selected cost models do not share a common capacity_max."
        )

    capex_mask = (
        selected_models["cost_type"] == "capex"
    )

    for column in [
        "coefficient",
        "exponent",
    ]:
        selected_models.loc[capex_mask, column] = pd.to_numeric(
            selected_models.loc[capex_mask, column],
            errors="coerce",
        )

        if selected_models.loc[capex_mask, column].isna().any():
            raise ValueError(
                f"CAPEX model column '{column}' contains invalid values."
            )

        if not np.isfinite(
            selected_models.loc[capex_mask, column]
        ).all():
            raise ValueError(
                f"CAPEX model column '{column}' contains non-finite values."
            )

    if (
        selected_models.loc[
            capex_mask,
            "coefficient",
        ] <= 0
    ).any():
        raise ValueError(
            "CAPEX power-model coefficient must be strictly positive."
        )

    if (
        selected_models.loc[
            capex_mask,
            "exponent",
        ] <= 0
    ).any():
        raise ValueError(
            "CAPEX power-model exponent must be strictly positive."
        )

    opex_mask = selected_models["cost_type"].isin(
        [
            "fixed_opex",
            "variable_opex",
        ]
    )

    for column in [
        "slope",
        "intercept",
    ]:
        selected_models.loc[opex_mask, column] = pd.to_numeric(
            selected_models.loc[opex_mask, column],
            errors="coerce",
        )

        if selected_models.loc[opex_mask, column].isna().any():
            raise ValueError(
                f"OPEX model column '{column}' contains invalid values."
            )

        if not np.isfinite(
            selected_models.loc[opex_mask, column]
        ).all():
            raise ValueError(
                f"OPEX model column '{column}' contains non-finite values."
            )

    cost_type_order = pd.CategoricalDtype(
        categories=[
            "capex",
            "fixed_opex",
            "variable_opex",
        ],
        ordered=True,
    )

    selected_models["cost_type"] = (
        selected_models["cost_type"]
        .astype(cost_type_order)
    )

    selected_models = (
        selected_models
        .sort_values("cost_type")
        .reset_index(drop=True)
    )

    selected_models["cost_type"] = (
        selected_models["cost_type"]
        .astype("string")
    )

    return selected_models


selected_h2_cost_models = load_selected_cost_models(
    cost_model_path=INPUT_SELECTED_COST_MODEL_PATH,
)

selected_capacity_min = (
    selected_h2_cost_models["capacity_min"].iloc[0]
)

selected_capacity_max = (
    selected_h2_cost_models["capacity_max"].iloc[0]
)

print(
    "Loaded selected H2 pipeline cost models: "
    f"{len(selected_h2_cost_models):,}"
)

print(
    "  Technology: "
    f"{selected_h2_cost_models['technology'].iloc[0]}"
)

print(
    "  Commodity:  "
    f"{selected_h2_cost_models['commodity'].iloc[0]}"
)

print(
    "  Capacity range: "
    f"{selected_capacity_min:,.0f} to "
    f"{selected_capacity_max:,.0f} "
    "t H2/year"
)

display(
    selected_h2_cost_models[
        [
            "cost_type",
            "model_type",
            "slope",
            "intercept",
            "coefficient",
            "exponent",
            "capacity_min",
            "capacity_max",
        ]
    ]
)

Loaded selected H2 pipeline cost models: 3
  Technology: H2_PIPE
  Commodity:  h2
  Capacity range: 53,980 to 1,727,345 t H2/year


,cost_type,model_type,slope,intercept,coefficient,exponent,capacity_min,capacity_max
0,capex,power,NaN,NaN,20885.563742,0.379112,53979.525993,1.727345e+06
1,fixed_opex,linear,0.144804,66041.133024,NaN,NaN,53979.525993,1.727345e+06
2,variable_opex,linear,0.180341,1087.392415,NaN,NaN,53979.525993,1.727345e+06


In [4]:
# =============================================================================
# Extract selected linear OPEX models
# =============================================================================

def extract_linear_opex_models(
    selected_models: pd.DataFrame,
) -> pd.DataFrame:
    """Extract fixed- and variable-OPEX linear regression coefficients.

    Parameters
    ----------
    selected_models : pd.DataFrame
        Selected pipeline cost models produced by Notebook 13.

    Returns
    -------
    pd.DataFrame
        One row each for fixed OPEX and variable OPEX, including their linear
        slope and intercept coefficients.

    Raises
    ------
    ValueError
        If either required OPEX model is missing or is not linear.
    """

    required_cost_types = [
        "fixed_opex",
        "variable_opex",
    ]

    opex_models = (
        selected_models.loc[
            selected_models["cost_type"].isin(required_cost_types)
        ]
        .copy()
        .sort_values("cost_type")
        .reset_index(drop=True)
    )

    if set(opex_models["cost_type"]) != set(required_cost_types):
        raise ValueError(
            "Selected models must contain exactly one fixed-OPEX model "
            "and one variable-OPEX model."
        )

    if not (opex_models["model_type"] == "linear").all():
        invalid_models = opex_models.loc[
            opex_models["model_type"] != "linear",
            [
                "cost_type",
                "model_type",
            ],
        ]

        raise ValueError(
            "Fixed and variable OPEX models must both be linear:\n"
            f"{invalid_models}"
        )

    for column in [
        "slope",
        "intercept",
    ]:
        opex_models[column] = pd.to_numeric(
            opex_models[column],
            errors="coerce",
        )

        if opex_models[column].isna().any():
            raise ValueError(
                f"OPEX model column '{column}' contains invalid values."
            )

    return opex_models[
        [
            "technology",
            "commodity",
            "cost_type",
            "model_type",
            "slope",
            "intercept",
            "capacity_min",
            "capacity_max",
            "currency",
            "currency_year",
            "cost_model_version",
        ]
    ]


selected_h2_opex_models = extract_linear_opex_models(
    selected_models=selected_h2_cost_models,
)

print(
    "Extracted selected H2 pipeline OPEX models: "
    f"{len(selected_h2_opex_models):,}"
)

display(selected_h2_opex_models)

Extracted selected H2 pipeline OPEX models: 2


,technology,commodity,cost_type,model_type,slope,intercept,capacity_min,capacity_max,currency,currency_year,cost_model_version
0,H2_PIPE,h2,fixed_opex,linear,0.144804,66041.133024,53979.525993,1.727345e+06,CAD,2020,v1
1,H2_PIPE,h2,variable_opex,linear,0.180341,1087.392415,53979.525993,1.727345e+06,CAD,2020,v1


In [5]:
# =============================================================================
# Build topology-free H2 pipeline ETLSegment template
# =============================================================================

ETL_SEGMENT_COUNT = 4
ETL_SPACING = "log"


def extract_capex_power_model(
    selected_models: pd.DataFrame,
) -> pd.Series:
    """Extract and validate the selected H2 pipeline CAPEX power model.

    Parameters
    ----------
    selected_models : pd.DataFrame
        Selected H2 pipeline regression models produced by Notebook 13.

    Returns
    -------
    pd.Series
        Selected CAPEX model containing its technology identifier, power-law
        coefficient and exponent, and valid capacity range.

    Raises
    ------
    ValueError
        If exactly one CAPEX power model is not available or its parameters
        are invalid.
    """

    capex_models = (
        selected_models.loc[
            selected_models["cost_type"] == "capex"
        ]
        .copy()
        .reset_index(drop=True)
    )

    if len(capex_models) != 1:
        raise ValueError(
            "Selected models must contain exactly one CAPEX model."
        )

    capex_model = capex_models.iloc[0].copy()

    if capex_model["model_type"] != "power":
        raise ValueError(
            "Selected CAPEX model must use a power regression, "
            f"but found '{capex_model['model_type']}'."
        )

    for column in [
        "coefficient",
        "exponent",
        "capacity_min",
        "capacity_max",
    ]:
        value = pd.to_numeric(
            capex_model[column],
            errors="coerce",
        )

        if pd.isna(value):
            raise ValueError(
                f"CAPEX model parameter '{column}' is invalid."
            )

        capex_model[column] = float(value)

    if capex_model["coefficient"] <= 0:
        raise ValueError(
            "CAPEX power-model coefficient must be positive."
        )

    if capex_model["exponent"] <= 0:
        raise ValueError(
            "CAPEX power-model exponent must be positive."
        )

    if capex_model["capacity_min"] <= 0:
        raise ValueError(
            "CAPEX model capacity_min must be positive."
        )

    if capex_model["capacity_max"] <= capex_model["capacity_min"]:
        raise ValueError(
            "CAPEX model capacity_max must be greater than capacity_min."
        )

    return capex_model


def build_h2_etlsegment_template(
    selected_models: pd.DataFrame,
    segment_count: int = ETL_SEGMENT_COUNT,
    spacing: str = ETL_SPACING,
) -> pd.DataFrame:
    """Build a topology-free H2 pipeline ETLSegment CAPEX template.

    The selected CAPEX power regression predicts total pipeline investment
    cost per kilometre as a function of annual pipeline capacity:

        cost_per_km = coefficient * capacity ** exponent

    Capacity breakpoints are constructed across the validated regression
    domain. Each ETL row stores the modelled CAPEX per kilometre at the lower
    and upper capacity bounds. Graph-edge regions and edge-distance scaling
    are added in a later mapping step.

    Parameters
    ----------
    selected_models : pd.DataFrame
        Selected H2 pipeline regression models.
    segment_count : int, default ETL_SEGMENT_COUNT
        Number of piecewise-linear capacity intervals to construct.
    spacing : str, default ETL_SPACING
        Breakpoint spacing method. Must be ``"log"`` or ``"linear"``.

    Returns
    -------
    pd.DataFrame
        Topology-free ETLSegment template with one row per capacity segment.

    Raises
    ------
    ValueError
        If the segment count or spacing method is invalid, or if the resulting
        segment table violates continuity and ordering requirements.
    """

    if segment_count < 1:
        raise ValueError(
            "ETL segment_count must be at least 1."
        )

    if spacing not in {
        "log",
        "linear",
    }:
        raise ValueError(
            "ETL spacing must be either 'log' or 'linear'."
        )

    capex_model = extract_capex_power_model(
        selected_models=selected_models,
    )

    capacity_min = capex_model["capacity_min"]
    capacity_max = capex_model["capacity_max"]
    coefficient = capex_model["coefficient"]
    exponent = capex_model["exponent"]

    breakpoint_count = segment_count + 1

    if spacing == "log":
        capacity_breakpoints = np.geomspace(
            capacity_min,
            capacity_max,
            breakpoint_count,
        )
    else:
        capacity_breakpoints = np.linspace(
            capacity_min,
            capacity_max,
            breakpoint_count,
        )

    cap_lower = capacity_breakpoints[:-1]
    cap_upper = capacity_breakpoints[1:]

    cost_lower_per_km = (
        coefficient
        * cap_lower ** exponent
    )

    cost_upper_per_km = (
        coefficient
        * cap_upper ** exponent
    )

    etl_template = pd.DataFrame(
        {
            "tech_or_group": capex_model["technology"],
            "segment": np.arange(
                segment_count,
                dtype=int,
            ),
            "cap_lower": cap_lower,
            "cap_upper": cap_upper,
            "cost_lower_per_km": cost_lower_per_km,
            "cost_upper_per_km": cost_upper_per_km,
            "data_id": DATA_ID,
        }
    )

    if not (
        etl_template["cap_upper"]
        > etl_template["cap_lower"]
    ).all():
        raise ValueError(
            "Every ETL segment must have cap_upper greater than cap_lower."
        )

    if not (
        etl_template["cost_upper_per_km"]
        > etl_template["cost_lower_per_km"]
    ).all():
        raise ValueError(
            "Every ETL segment must have cost_upper_per_km greater than "
            "cost_lower_per_km."
        )

    if len(etl_template) > 1:
        if not np.allclose(
            etl_template["cap_upper"].iloc[:-1],
            etl_template["cap_lower"].iloc[1:],
        ):
            raise ValueError(
                "ETL capacity segments are not contiguous."
            )

        if not np.allclose(
            etl_template["cost_upper_per_km"].iloc[:-1],
            etl_template["cost_lower_per_km"].iloc[1:],
        ):
            raise ValueError(
                "ETL CAPEX bounds are not contiguous."
            )

    if not np.isclose(
        etl_template["cap_lower"].iloc[0],
        capacity_min,
    ):
        raise ValueError(
            "First ETL segment does not begin at model capacity_min."
        )

    if not np.isclose(
        etl_template["cap_upper"].iloc[-1],
        capacity_max,
    ):
        raise ValueError(
            "Final ETL segment does not end at model capacity_max."
        )

    return etl_template


h2_etlsegment_template = build_h2_etlsegment_template(
    selected_models=selected_h2_cost_models,
    segment_count=ETL_SEGMENT_COUNT,
    spacing=ETL_SPACING,
)

print("Built topology-free H2 pipeline ETLSegment template.")
print(f"  Segments:       {len(h2_etlsegment_template):,}")
print(f"  Spacing:        {ETL_SPACING}")
print(
    "  Capacity range: "
    f"{h2_etlsegment_template['cap_lower'].min():,.0f} to "
    f"{h2_etlsegment_template['cap_upper'].max():,.0f} "
    "t H2/year"
)
print(
    "  CAPEX range:   "
    f"${h2_etlsegment_template['cost_lower_per_km'].min():,.0f} to "
    f"${h2_etlsegment_template['cost_upper_per_km'].max():,.0f} "
    "CAD2020/km"
)

display(h2_etlsegment_template)

Built topology-free H2 pipeline ETLSegment template.
  Segments:       4
  Spacing:        log
  Capacity range: 53,980 to 1,727,345 t H2/year
  CAPEX range:   $1,299,844 to $4,836,282 CAD2020/km


,tech_or_group,segment,cap_lower,cap_upper,cost_lower_per_km,cost_upper_per_km,data_id
0,H2_PIPE,0,53979.525993,1.283857e+05,1.299844e+06,1.805287e+06,GEO001
1,H2_PIPE,1,128385.672751,3.053543e+05,1.805287e+06,2.507272e+06,GEO001
2,H2_PIPE,2,305354.311000,7.262590e+05,2.507272e+06,3.482223e+06,GEO001
3,H2_PIPE,3,726259.038475,1.727345e+06,3.482223e+06,4.836282e+06,GEO001


In [6]:
# =============================================================================
# Discover graph-edge input tables
# =============================================================================

def discover_graph_edge_tables(
    graph_directory: Path,
    filename_pattern: str,
) -> list[Path]:
    """Discover processed graph-edge CSV files.

    Parameters
    ----------
    graph_directory : Path
        Directory containing processed graph products.
    filename_pattern : str
        Glob pattern used to identify graph-edge CSV files.

    Returns
    -------
    list[Path]
        Sorted list of matching graph-edge table paths.

    Raises
    ------
    FileNotFoundError
        If the graph directory does not exist or no matching files are found.
    """

    if not graph_directory.exists():
        raise FileNotFoundError(
            "Processed graph directory not found: "
            f"{graph_directory}"
        )

    graph_edge_paths = sorted(
        graph_directory.glob(filename_pattern)
    )

    if not graph_edge_paths:
        raise FileNotFoundError(
            "No graph-edge CSV files were found in "
            f"{graph_directory} using pattern "
            f"'{filename_pattern}'."
        )

    return graph_edge_paths


graph_edge_paths = discover_graph_edge_tables(
    graph_directory=PROCESSED_GRAPH_DIR,
    filename_pattern=GRAPH_EDGE_PATTERN,
)

DEVELOPMENT_GRAPH_NAME = (
    "canada_basemap_0.5deg_centroid_graph_edges.csv"
)

development_graph_path = (
    PROCESSED_GRAPH_DIR
    / DEVELOPMENT_GRAPH_NAME
)

if development_graph_path not in graph_edge_paths:
    raise FileNotFoundError(
        "Development graph-edge table not found: "
        f"{development_graph_path}"
    )

print(
    f"Discovered graph-edge tables: "
    f"{len(graph_edge_paths):,}"
)

for graph_path in graph_edge_paths:
    marker = (
        "  <- development case"
        if graph_path == development_graph_path
        else ""
    )

    print(
        f"  - {graph_path.name}"
        f"{marker}"
    )

Discovered graph-edge tables: 12
  - canada_basemap_0.5deg_centroid_graph_edges.csv  <- development case
  - canada_basemap_0.5deg_intersects_graph_edges.csv
  - canada_basemap_1deg_centroid_graph_edges.csv
  - canada_basemap_1deg_intersects_graph_edges.csv
  - canada_basemap_2deg_centroid_graph_edges.csv
  - canada_basemap_2deg_intersects_graph_edges.csv
  - canada_basemap_3deg_centroid_graph_edges.csv
  - canada_basemap_3deg_intersects_graph_edges.csv
  - canada_basemap_4deg_centroid_graph_edges.csv
  - canada_basemap_4deg_intersects_graph_edges.csv
  - canada_basemap_5deg_centroid_graph_edges.csv
  - canada_basemap_5deg_intersects_graph_edges.csv


In [7]:
# =============================================================================
# Load development graph-edge table
# =============================================================================

def load_graph_edge_table(
    graph_edge_path: Path,
) -> pd.DataFrame:
    """Load a processed graph-edge CSV table.

    Parameters
    ----------
    graph_edge_path : Path
        Path to a processed graph-edge CSV.

    Returns
    -------
    pd.DataFrame
        Loaded graph-edge table.

    Raises
    ------
    FileNotFoundError
        If the graph-edge file does not exist.
    ValueError
        If the loaded table is empty.
    """

    if not graph_edge_path.exists():
        raise FileNotFoundError(
            "Graph-edge table not found: "
            f"{graph_edge_path}"
        )

    graph_edges = pd.read_csv(
        graph_edge_path,
        encoding="utf-8",
    )

    if graph_edges.empty:
        raise ValueError(
            "Graph-edge table is empty: "
            f"{graph_edge_path}"
        )

    return graph_edges


development_graph_edges = load_graph_edge_table(
    graph_edge_path=development_graph_path,
)

print(
    "Loaded development graph-edge table: "
    f"{development_graph_edges.shape[0]:,} rows × "
    f"{development_graph_edges.shape[1]:,} columns"
)

display(
    development_graph_edges.head()
)

Loaded development graph-edge table: 24,670 rows × 11 columns


,edge_region,region_from,region_to,direction,lon_from,lat_from,lon_to,lat_to,distance_km,resolution_deg,keep_method
0,R0-R1,R0,R1,right,-82.75,42.25,-82.25,42.25,41.262779,0.5,centroid
1,R1-R2,R1,R2,up,-82.25,42.25,-82.25,42.75,55.541501,0.5,centroid
2,R1-R0,R1,R0,left,-82.25,42.25,-82.75,42.25,41.262779,0.5,centroid
3,R2-R1,R2,R1,down,-82.25,42.75,-82.25,42.25,55.541501,0.5,centroid
4,R2-R3,R2,R3,right,-82.25,42.75,-81.75,42.75,40.935327,0.5,centroid


In [8]:
# =============================================================================
# Encode H2 ETLSegment rows for development graph edges
# =============================================================================

ETLSEGMENT_COLUMNS = [
    "region",
    "tech_or_group",
    "segment",
    "cap_lower",
    "cap_upper",
    "cost_lower",
    "cost_upper",
    "data_id",
]


def encode_h2_etlsegment_for_edges(
    graph_edges: pd.DataFrame,
    etlsegment_template: pd.DataFrame,
) -> pd.DataFrame:
    """Encode the H2 ETLSegment template for candidate graph edges.

    The graph ``edge_region`` identifier becomes the TEMOA ``region`` field
    because transport technologies are encoded on edge pseudo-regions.
    Edge distance scales the per-kilometre CAPEX bounds into total edge costs.

    Parameters
    ----------
    graph_edges : pd.DataFrame
        Candidate graph edges containing ``edge_region`` and ``distance_km``.
    etlsegment_template : pd.DataFrame
        Topology-free H2 pipeline segment template containing per-kilometre
        CAPEX bounds.

    Returns
    -------
    pd.DataFrame
        SQL-aligned H2 pipeline ETLSegment rows.
    """

    required_graph_columns = [
        "edge_region",
        "distance_km",
    ]

    required_template_columns = [
        "tech_or_group",
        "segment",
        "cap_lower",
        "cap_upper",
        "cost_lower_per_km",
        "cost_upper_per_km",
        "data_id",
    ]

    missing_graph_columns = [
        column
        for column in required_graph_columns
        if column not in graph_edges.columns
    ]

    missing_template_columns = [
        column
        for column in required_template_columns
        if column not in etlsegment_template.columns
    ]

    if missing_graph_columns:
        raise ValueError(
            "Graph-edge table is missing required columns: "
            f"{missing_graph_columns}"
        )

    if missing_template_columns:
        raise ValueError(
            "ETLSegment template is missing required columns: "
            f"{missing_template_columns}"
        )

    edge_cost_scalars = (
        graph_edges[
            [
                "edge_region",
                "distance_km",
            ]
        ]
        .copy()
    )

    edge_cost_scalars["edge_region"] = (
        edge_cost_scalars["edge_region"]
        .astype("string")
        .str.strip()
    )

    edge_cost_scalars["distance_km"] = pd.to_numeric(
        edge_cost_scalars["distance_km"],
        errors="coerce",
    )

    if edge_cost_scalars["edge_region"].isna().any():
        raise ValueError(
            "Graph-edge table contains missing edge_region values."
        )

    if (edge_cost_scalars["edge_region"] == "").any():
        raise ValueError(
            "Graph-edge table contains blank edge_region values."
        )

    if edge_cost_scalars["edge_region"].duplicated().any():
        raise ValueError(
            "Graph-edge table contains duplicate edge_region values."
        )

    if edge_cost_scalars["distance_km"].isna().any():
        raise ValueError(
            "Graph-edge table contains invalid distance_km values."
        )

    if not np.isfinite(edge_cost_scalars["distance_km"]).all():
        raise ValueError(
            "Graph-edge table contains non-finite distance_km values."
        )

    if (edge_cost_scalars["distance_km"] <= 0).any():
        raise ValueError(
            "Graph-edge distances must be strictly positive."
        )

    encoded_segments = edge_cost_scalars.merge(
        etlsegment_template,
        how="cross",
    )

    encoded_segments["cost_lower"] = (
        encoded_segments["cost_lower_per_km"]
        * encoded_segments["distance_km"]
    )

    encoded_segments["cost_upper"] = (
        encoded_segments["cost_upper_per_km"]
        * encoded_segments["distance_km"]
    )

    # TEMOA calls both node IDs and transport edge pseudo-region IDs "region".
    encoded_segments = encoded_segments.rename(
        columns={
            "edge_region": "region",
        }
    )

    encoded_segments = encoded_segments[
        ETLSEGMENT_COLUMNS
    ].copy()

    expected_rows = (
        graph_edges["edge_region"].nunique()
        * len(etlsegment_template)
    )

    if len(encoded_segments) != expected_rows:
        raise ValueError(
            "Encoded ETLSegment row count is incorrect: "
            f"expected {expected_rows:,}, found "
            f"{len(encoded_segments):,}."
        )

    duplicate_rows = encoded_segments.duplicated(
        subset=[
            "region",
            "tech_or_group",
            "segment",
        ],
        keep=False,
    )

    if duplicate_rows.any():
        raise ValueError(
            "Encoded ETLSegment table contains duplicate "
            "edge-region, technology, and segment combinations."
        )

    return (
        encoded_segments
        .sort_values(
            [
                "region",
                "tech_or_group",
                "segment",
            ]
        )
        .reset_index(drop=True)
    )


mapped_h2_etlsegment = encode_h2_etlsegment_for_edges(
    graph_edges=development_graph_edges,
    etlsegment_template=h2_etlsegment_template,
)

print("Encoded H2 ETLSegment costs for development graph edges.")
print(
    "  Edge regions:      "
    f"{development_graph_edges['edge_region'].nunique():,}"
)
print(
    "  Segments per edge: "
    f"{len(h2_etlsegment_template):,}"
)
print(
    "  Total ETLSegment rows: "
    f"{len(mapped_h2_etlsegment):,}"
)
print(
    "  Output columns:    "
    f"{mapped_h2_etlsegment.columns.tolist()}"
)

display(mapped_h2_etlsegment.head(12))

Encoded H2 ETLSegment costs for development graph edges.
  Edge regions:      24,670
  Segments per edge: 4
  Total ETLSegment rows: 98,680
  Output columns:    ['region', 'tech_or_group', 'segment', 'cap_lower', 'cap_upper', 'cost_lower', 'cost_upper', 'data_id']


,region,tech_or_group,segment,cap_lower,cap_upper,cost_lower,cost_upper,data_id
0,R0-R1,H2_PIPE,0,53979.525993,1.283857e+05,5.363517e+07,7.449117e+07,GEO001
1,R0-R1,H2_PIPE,1,128385.672751,3.053543e+05,7.449117e+07,1.034570e+08,GEO001
2,R0-R1,H2_PIPE,2,305354.311000,7.262590e+05,1.034570e+08,1.436862e+08,GEO001
3,R0-R1,H2_PIPE,3,726259.038475,1.727345e+06,1.436862e+08,1.995585e+08,GEO001
4,R1-R0,H2_PIPE,0,53979.525993,1.283857e+05,5.363517e+07,7.449117e+07,GEO001
5,R1-R0,H2_PIPE,1,128385.672751,3.053543e+05,7.449117e+07,1.034570e+08,GEO001
6,R1-R0,H2_PIPE,2,305354.311000,7.262590e+05,1.034570e+08,1.436862e+08,GEO001
7,R1-R0,H2_PIPE,3,726259.038475,1.727345e+06,1.436862e+08,1.995585e+08,GEO001
8,R1-R2,H2_PIPE,0,53979.525993,1.283857e+05,7.219528e+07,1.002684e+08,GEO001
9,R1-R2,H2_PIPE,1,128385.672751,3.053543e+05,1.002684e+08,1.392576e+08,GEO001


In [9]:
# =============================================================================
# Validate encoded H2 ETLSegment table
# =============================================================================

def validate_encoded_h2_etlsegment(
    encoded_segments: pd.DataFrame,
    graph_edges: pd.DataFrame,
    segment_count: int,
) -> None:
    """Validate edge coverage, uniqueness, and segment continuity."""

    expected_regions = set(
        graph_edges["edge_region"].astype(str)
    )

    actual_regions = set(
        encoded_segments["region"].astype(str)
    )

    if actual_regions != expected_regions:
        missing_regions = sorted(
            expected_regions - actual_regions
        )

        unexpected_regions = sorted(
            actual_regions - expected_regions
        )

        raise ValueError(
            "Encoded ETLSegment edge coverage differs from the graph.\n"
            f"Missing regions: {missing_regions[:10]}\n"
            f"Unexpected regions: {unexpected_regions[:10]}"
        )

    segment_counts = (
        encoded_segments
        .groupby(
            [
                "region",
                "tech_or_group",
            ]
        )["segment"]
        .nunique()
    )

    if not (segment_counts == segment_count).all():
        invalid_counts = segment_counts.loc[
            segment_counts != segment_count
        ]

        raise ValueError(
            "Some edge regions do not contain the expected number "
            f"of ETL segments:\n{invalid_counts.head(10)}"
        )

    duplicate_rows = encoded_segments.duplicated(
        subset=[
            "region",
            "tech_or_group",
            "segment",
        ],
        keep=False,
    )

    if duplicate_rows.any():
        raise ValueError(
            "Encoded ETLSegment table contains duplicate "
            "region-technology-segment rows."
        )

    ordered = (
        encoded_segments
        .sort_values(
            [
                "region",
                "tech_or_group",
                "segment",
            ]
        )
        .reset_index(drop=True)
    )

    grouped = ordered.groupby(
        [
            "region",
            "tech_or_group",
        ],
        sort=False,
    )

    for group_key, group in grouped:
        group = group.sort_values("segment")

        expected_segment_ids = list(
            range(segment_count)
        )

        actual_segment_ids = (
            group["segment"]
            .astype(int)
            .tolist()
        )

        if actual_segment_ids != expected_segment_ids:
            raise ValueError(
                f"Invalid segment numbering for {group_key}: "
                f"{actual_segment_ids}"
            )

        if not np.allclose(
            group["cap_upper"].iloc[:-1],
            group["cap_lower"].iloc[1:],
        ):
            raise ValueError(
                f"Capacity bounds are not contiguous for {group_key}."
            )

        if not np.allclose(
            group["cost_upper"].iloc[:-1],
            group["cost_lower"].iloc[1:],
        ):
            raise ValueError(
                f"Cost bounds are not contiguous for {group_key}."
            )

    print("Encoded H2 ETLSegment validation passed.")
    print(f"  Edge coverage:     {len(actual_regions):,}")
    print(f"  Segments per edge: {segment_count:,}")
    print(f"  Total rows:        {len(encoded_segments):,}")
    print("  Duplicate rows:    0")
    print("  Capacity bounds:   contiguous")
    print("  Cost bounds:       contiguous")


validate_encoded_h2_etlsegment(
    encoded_segments=mapped_h2_etlsegment,
    graph_edges=development_graph_edges,
    segment_count=ETL_SEGMENT_COUNT,
)

Encoded H2 ETLSegment validation passed.
  Edge coverage:     24,670
  Segments per edge: 4
  Total rows:        98,680
  Duplicate rows:    0
  Capacity bounds:   contiguous
  Cost bounds:       contiguous


In [10]:
# =============================================================================
# Build topology-free H2 pipeline fixed and variable OPEX coefficient template
# =============================================================================

H2_PIPELINE_OPEX_COEFFICIENT_COLUMNS = [
    "technology",
    "commodity",
    "cost_type",
    "coefficient_per_km",
    "intercept_cost",
    "units",
    "notes",
    "data_source",
    "dq_cred",
    "dq_geog",
    "dq_struc",
    "dq_tech",
    "dq_time",
    "data_id",
]


def build_h2_pipeline_opex_coefficient_template(
    selected_models: pd.DataFrame,
) -> pd.DataFrame:
    """Extract topology-free fixed and variable H2 pipeline OPEX models.

    This table contains two distinct operating-cost models:

        fixed_opex
            Mapped to the Temoa CostFixed table.

        variable_opex
            Mapped to the Temoa CostVariable table.

    Both selected models are linear:

        cost = coefficient_per_km * distance_km + intercept_cost

    Graph-edge distance is applied later when constructing the final
    CostFixed and CostVariable rows.
    """

    required_columns = [
        "technology",
        "commodity",
        "cost_type",
        "model_type",
        "slope",
        "intercept",
    ]

    missing_columns = [
        column
        for column in required_columns
        if column not in selected_models.columns
    ]

    if missing_columns:
        raise ValueError(
            "Selected cost models are missing required columns: "
            f"{missing_columns}"
        )

    pipeline_opex_models = selected_models.loc[
        selected_models["cost_type"].isin(
            [
                "fixed_opex",
                "variable_opex",
            ]
        )
    ].copy()

    expected_cost_types = {
        "fixed_opex",
        "variable_opex",
    }

    actual_cost_types = set(
        pipeline_opex_models["cost_type"]
    )

    if actual_cost_types != expected_cost_types:
        raise ValueError(
            "Expected exactly fixed_opex and variable_opex models, "
            f"but found {sorted(actual_cost_types)}."
        )

    if len(pipeline_opex_models) != 2:
        raise ValueError(
            "Expected exactly one fixed_opex row and one "
            "variable_opex row."
        )

    if pipeline_opex_models["cost_type"].duplicated().any():
        raise ValueError(
            "Selected pipeline OPEX models contain duplicate "
            "cost_type rows."
        )

    if not (
        pipeline_opex_models["model_type"] == "linear"
    ).all():
        raise ValueError(
            "H2 pipeline fixed and variable OPEX models must be linear."
        )

    if pipeline_opex_models["technology"].nunique() != 1:
        raise ValueError(
            "Selected pipeline OPEX models must refer to one technology."
        )

    if pipeline_opex_models["commodity"].nunique() != 1:
        raise ValueError(
            "Selected pipeline OPEX models must refer to one commodity."
        )

    if pipeline_opex_models["technology"].iloc[0] != "H2_PIPE":
        raise ValueError(
            "Expected H2 pipeline technology 'H2_PIPE', but found "
            f"'{pipeline_opex_models['technology'].iloc[0]}'."
        )

    if pipeline_opex_models["commodity"].iloc[0] != "h2":
        raise ValueError(
            "Expected H2 commodity 'h2', but found "
            f"'{pipeline_opex_models['commodity'].iloc[0]}'."
        )

    for column in [
        "slope",
        "intercept",
    ]:
        pipeline_opex_models[column] = pd.to_numeric(
            pipeline_opex_models[column],
            errors="coerce",
        )

        if pipeline_opex_models[column].isna().any():
            raise ValueError(
                f"Pipeline OPEX column '{column}' contains invalid values."
            )

        if not np.isfinite(
            pipeline_opex_models[column]
        ).all():
            raise ValueError(
                f"Pipeline OPEX column '{column}' contains "
                "non-finite values."
            )

    pipeline_opex_template = pipeline_opex_models.rename(
        columns={
            "slope": "coefficient_per_km",
            "intercept": "intercept_cost",
        }
    )

    pipeline_opex_template["units"] = (
        pipeline_opex_template["cost_type"].map(
            {
                "fixed_opex": (
                    "CAD2020/(t H2 capacity/year)/km"
                ),
                "variable_opex": (
                    "CAD2020/t H2/km"
                ),
            }
        )
    )

    pipeline_opex_template["notes"] = (
        pipeline_opex_template["cost_type"].map(
            {
                "fixed_opex": (
                    "Selected linear fixed operating-cost model for "
                    "H2 pipeline transport. Mapped to Temoa CostFixed "
                    "using the inherited geospatial equation: "
                    "cost = coefficient_per_km * distance_km "
                    "+ intercept_cost."
                ),
                "variable_opex": (
                    "Selected linear variable operating-cost model for "
                    "H2 pipeline transport. Mapped to Temoa CostVariable "
                    "using the inherited geospatial equation: "
                    "cost = coefficient_per_km * distance_km "
                    "+ intercept_cost."
                ),
            }
        )
    )

    pipeline_opex_template["data_source"] = (
        "Transition Accelerator H2 pipeline cost model"
    )

    pipeline_opex_template["dq_cred"] = None
    pipeline_opex_template["dq_geog"] = None
    pipeline_opex_template["dq_struc"] = None
    pipeline_opex_template["dq_tech"] = None
    pipeline_opex_template["dq_time"] = None
    pipeline_opex_template["data_id"] = DATA_ID

    pipeline_opex_template = (
        pipeline_opex_template[
            H2_PIPELINE_OPEX_COEFFICIENT_COLUMNS
        ]
        .sort_values("cost_type")
        .reset_index(drop=True)
    )

    return pipeline_opex_template


h2_pipeline_opex_coefficient_template = (
    build_h2_pipeline_opex_coefficient_template(
        selected_models=selected_h2_cost_models,
    )
)

print(
    "Built topology-free H2 pipeline fixed and variable "
    "OPEX coefficient template."
)
print(
    "  Cost types:  "
    f"{h2_pipeline_opex_coefficient_template['cost_type'].tolist()}"
)
print(
    "  Technology:  "
    f"{h2_pipeline_opex_coefficient_template['technology'].iloc[0]}"
)
print(
    "  Commodity:   "
    f"{h2_pipeline_opex_coefficient_template['commodity'].iloc[0]}"
)
print(
    "  Data source: "
    f"{h2_pipeline_opex_coefficient_template['data_source'].iloc[0]}"
)

display(h2_pipeline_opex_coefficient_template)

Built topology-free H2 pipeline fixed and variable OPEX coefficient template.
  Cost types:  ['fixed_opex', 'variable_opex']
  Technology:  H2_PIPE
  Commodity:   h2
  Data source: Transition Accelerator H2 pipeline cost model


,technology,commodity,cost_type,coefficient_per_km,intercept_cost,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,H2_PIPE,h2,fixed_opex,0.144804,66041.133024,CAD2020/(t H2 capacity/year)/km,Selected linear fixed operating-cost model for...,Transition Accelerator H2 pipeline cost model,None,None,None,None,None,GEO001
1,H2_PIPE,h2,variable_opex,0.180341,1087.392415,CAD2020/t H2/km,Selected linear variable operating-cost model ...,Transition Accelerator H2 pipeline cost model,None,None,None,None,None,GEO001


In [11]:
# =============================================================================
# Map H2 pipeline fixed and variable OPEX onto development graph edges
# =============================================================================

COSTFIXED_COLUMNS = [
    "region",
    "period",
    "tech",
    "vintage",
    "cost",
    "units",
    "notes",
    "data_source",
    "dq_cred",
    "dq_geog",
    "dq_struc",
    "dq_tech",
    "dq_time",
    "data_id",
]

COSTVARIABLE_COLUMNS = COSTFIXED_COLUMNS.copy()


def map_h2_pipeline_opex_to_graph_edges(
    graph_edges: pd.DataFrame,
    opex_template: pd.DataFrame,
    period: int = 1,
    vintage: int = 1,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Map topology-free H2 pipeline fixed and variable OPEX models.

    The two cost types are mapped separately:

        fixed_opex
            Produces CostFixed rows.

        variable_opex
            Produces CostVariable rows.

    Edge-specific costs follow the inherited Geospatial-CANOE
    transport-cost convention:

        cost = intercept_cost
               + coefficient_per_km * distance_km

    The returned DataFrames match the existing CostFixed and
    CostVariable table schemas exactly.
    """

    required_graph_columns = [
        "edge_region",
        "distance_km",
    ]

    required_opex_columns = [
        "technology",
        "commodity",
        "cost_type",
        "coefficient_per_km",
        "intercept_cost",
        "units",
        "notes",
        "data_source",
        "dq_cred",
        "dq_geog",
        "dq_struc",
        "dq_tech",
        "dq_time",
        "data_id",
    ]

    missing_graph_columns = [
        column
        for column in required_graph_columns
        if column not in graph_edges.columns
    ]

    missing_opex_columns = [
        column
        for column in required_opex_columns
        if column not in opex_template.columns
    ]

    if missing_graph_columns:
        raise ValueError(
            "Graph-edge table is missing required columns: "
            f"{missing_graph_columns}"
        )

    if missing_opex_columns:
        raise ValueError(
            "Pipeline OPEX coefficient template is missing required "
            f"columns: {missing_opex_columns}"
        )

    expected_cost_types = {
        "fixed_opex",
        "variable_opex",
    }

    actual_cost_types = set(
        opex_template["cost_type"]
    )

    if actual_cost_types != expected_cost_types:
        raise ValueError(
            "Pipeline OPEX coefficient template must contain exactly "
            f"{sorted(expected_cost_types)}, but contains "
            f"{sorted(actual_cost_types)}."
        )

    if len(opex_template) != 2:
        raise ValueError(
            "Expected exactly one fixed_opex row and one "
            "variable_opex row."
        )

    if opex_template["cost_type"].duplicated().any():
        raise ValueError(
            "Pipeline OPEX coefficient template contains duplicate "
            "cost_type rows."
        )

    if opex_template["technology"].nunique() != 1:
        raise ValueError(
            "Pipeline OPEX coefficient template must contain one "
            "technology."
        )

    if opex_template["technology"].iloc[0] != "H2_PIPE":
        raise ValueError(
            "Expected pipeline OPEX technology 'H2_PIPE', but found "
            f"'{opex_template['technology'].iloc[0]}'."
        )

    if opex_template["commodity"].nunique() != 1:
        raise ValueError(
            "Pipeline OPEX coefficient template must contain one "
            "commodity."
        )

    if opex_template["commodity"].iloc[0] != "h2":
        raise ValueError(
            "Expected pipeline OPEX commodity 'h2', but found "
            f"'{opex_template['commodity'].iloc[0]}'."
        )

    edge_frame = (
        graph_edges[
            [
                "edge_region",
                "distance_km",
            ]
        ]
        .copy()
        .rename(
            columns={
                "edge_region": "region",
            }
        )
    )

    edge_frame["region"] = (
        edge_frame["region"]
        .astype("string")
        .str.strip()
    )

    edge_frame["distance_km"] = pd.to_numeric(
        edge_frame["distance_km"],
        errors="coerce",
    )

    if edge_frame["region"].isna().any():
        raise ValueError(
            "Graph-edge table contains missing edge_region values."
        )

    if (edge_frame["region"] == "").any():
        raise ValueError(
            "Graph-edge table contains blank edge_region values."
        )

    if edge_frame["region"].duplicated().any():
        duplicate_regions = (
            edge_frame.loc[
                edge_frame["region"].duplicated(
                    keep=False
                ),
                "region",
            ]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(
            "Graph-edge table contains duplicate edge_region values: "
            f"{duplicate_regions[:10]}"
        )

    if edge_frame["distance_km"].isna().any():
        raise ValueError(
            "Graph-edge table contains invalid distance_km values."
        )

    if not np.isfinite(
        edge_frame["distance_km"]
    ).all():
        raise ValueError(
            "Graph-edge table contains non-finite distance_km values."
        )

    if (
        edge_frame["distance_km"] <= 0
    ).any():
        raise ValueError(
            "Graph-edge distances must be strictly positive."
        )

    opex_frame = opex_template.copy()

    for column in [
        "coefficient_per_km",
        "intercept_cost",
    ]:
        opex_frame[column] = pd.to_numeric(
            opex_frame[column],
            errors="coerce",
        )

        if opex_frame[column].isna().any():
            raise ValueError(
                f"Pipeline OPEX coefficient column '{column}' "
                "contains invalid values."
            )

        if not np.isfinite(
            opex_frame[column]
        ).all():
            raise ValueError(
                f"Pipeline OPEX coefficient column '{column}' "
                "contains non-finite values."
            )

    mapped_opex = edge_frame.merge(
        opex_frame,
        how="cross",
    )

    mapped_opex["cost"] = (
        mapped_opex["intercept_cost"]
        + mapped_opex["coefficient_per_km"]
        * mapped_opex["distance_km"]
    )

    if mapped_opex["cost"].isna().any():
        raise ValueError(
            "Mapped H2 pipeline OPEX contains missing cost values."
        )

    if not np.isfinite(
        mapped_opex["cost"]
    ).all():
        raise ValueError(
            "Mapped H2 pipeline OPEX contains non-finite cost values."
        )

    if (
        mapped_opex["cost"] < 0
    ).any():
        invalid_rows = mapped_opex.loc[
            mapped_opex["cost"] < 0,
            [
                "region",
                "cost_type",
                "distance_km",
                "coefficient_per_km",
                "intercept_cost",
                "cost",
            ],
        ]

        raise ValueError(
            "Mapped H2 pipeline OPEX contains negative costs:\n"
            f"{invalid_rows.head(10)}"
        )

    mapped_opex["period"] = period
    mapped_opex["tech"] = mapped_opex["technology"]
    mapped_opex["vintage"] = vintage

    mapped_h2_costfixed = (
        mapped_opex.loc[
            mapped_opex["cost_type"] == "fixed_opex",
            COSTFIXED_COLUMNS,
        ]
        .sort_values(
            [
                "region",
                "period",
                "tech",
                "vintage",
            ]
        )
        .reset_index(drop=True)
    )

    mapped_h2_costvariable = (
        mapped_opex.loc[
            mapped_opex["cost_type"] == "variable_opex",
            COSTVARIABLE_COLUMNS,
        ]
        .sort_values(
            [
                "region",
                "period",
                "tech",
                "vintage",
            ]
        )
        .reset_index(drop=True)
    )

    expected_rows = len(edge_frame)

    if len(mapped_h2_costfixed) != expected_rows:
        raise ValueError(
            "H2 CostFixed row count is incorrect: "
            f"expected {expected_rows:,}, found "
            f"{len(mapped_h2_costfixed):,}."
        )

    if len(mapped_h2_costvariable) != expected_rows:
        raise ValueError(
            "H2 CostVariable row count is incorrect: "
            f"expected {expected_rows:,}, found "
            f"{len(mapped_h2_costvariable):,}."
        )

    schema_key_columns = [
        "region",
        "period",
        "tech",
        "vintage",
    ]

    if mapped_h2_costfixed.duplicated(
        subset=schema_key_columns
    ).any():
        raise ValueError(
            "Mapped H2 CostFixed contains duplicate schema keys."
        )

    if mapped_h2_costvariable.duplicated(
        subset=schema_key_columns
    ).any():
        raise ValueError(
            "Mapped H2 CostVariable contains duplicate schema keys."
        )

    if list(
        mapped_h2_costfixed.columns
    ) != COSTFIXED_COLUMNS:
        raise ValueError(
            "Mapped H2 CostFixed columns do not match the "
            "required schema."
        )

    if list(
        mapped_h2_costvariable.columns
    ) != COSTVARIABLE_COLUMNS:
        raise ValueError(
            "Mapped H2 CostVariable columns do not match the "
            "required schema."
        )

    return (
        mapped_h2_costfixed,
        mapped_h2_costvariable,
    )


(
    mapped_h2_costfixed,
    mapped_h2_costvariable,
) = map_h2_pipeline_opex_to_graph_edges(
    graph_edges=development_graph_edges,
    opex_template=h2_pipeline_opex_coefficient_template,
)

print(
    "Mapped H2 pipeline fixed and variable OPEX onto "
    "development graph edges."
)
print(
    "  Edge regions:       "
    f"{development_graph_edges['edge_region'].nunique():,}"
)
print(
    "  CostFixed rows:      "
    f"{len(mapped_h2_costfixed):,}"
)
print(
    "  CostVariable rows:   "
    f"{len(mapped_h2_costvariable):,}"
)
print(
    "  CostFixed range:     "
    f"{mapped_h2_costfixed['cost'].min():,.6f} to "
    f"{mapped_h2_costfixed['cost'].max():,.6f}"
)
print(
    "  CostVariable range:  "
    f"{mapped_h2_costvariable['cost'].min():,.6f} to "
    f"{mapped_h2_costvariable['cost'].max():,.6f}"
)
print(
    "  CostFixed columns:   "
    f"{mapped_h2_costfixed.columns.tolist()}"
)
print(
    "  CostVariable columns:"
    f" {mapped_h2_costvariable.columns.tolist()}"
)

display(mapped_h2_costfixed.head())
display(mapped_h2_costvariable.head())

Mapped H2 pipeline fixed and variable OPEX onto development graph edges.
  Edge regions:       24,670
  CostFixed rows:      24,670
  CostVariable rows:   24,670
  CostFixed range:     66,042.153522 to 66,049.218509
  CostVariable range:  1,088.663353 to 1,097.462156
  CostFixed columns:   ['region', 'period', 'tech', 'vintage', 'cost', 'units', 'notes', 'data_source', 'dq_cred', 'dq_geog', 'dq_struc', 'dq_tech', 'dq_time', 'data_id']
  CostVariable columns: ['region', 'period', 'tech', 'vintage', 'cost', 'units', 'notes', 'data_source', 'dq_cred', 'dq_geog', 'dq_struc', 'dq_tech', 'dq_time', 'data_id']


,region,period,tech,vintage,cost,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,R0-R1,1,H2_PIPE,1,66047.108047,CAD2020/(t H2 capacity/year)/km,Selected linear fixed operating-cost model for...,Transition Accelerator H2 pipeline cost model,None,None,None,None,None,GEO001
1,R1-R0,1,H2_PIPE,1,66047.108047,CAD2020/(t H2 capacity/year)/km,Selected linear fixed operating-cost model for...,Transition Accelerator H2 pipeline cost model,None,None,None,None,None,GEO001
2,R1-R2,1,H2_PIPE,1,66049.175666,CAD2020/(t H2 capacity/year)/km,Selected linear fixed operating-cost model for...,Transition Accelerator H2 pipeline cost model,None,None,None,None,None,GEO001
3,R10-R11,1,H2_PIPE,1,66046.964439,CAD2020/(t H2 capacity/year)/km,Selected linear fixed operating-cost model for...,Transition Accelerator H2 pipeline cost model,None,None,None,None,None,GEO001
4,R10-R17,1,H2_PIPE,1,66049.177783,CAD2020/(t H2 capacity/year)/km,Selected linear fixed operating-cost model for...,Transition Accelerator H2 pipeline cost model,None,None,None,None,None,GEO001


,region,period,tech,vintage,cost,units,notes,data_source,dq_cred,dq_geog,dq_struc,dq_tech,dq_time,data_id
0,R0-R1,1,H2_PIPE,1,1094.833766,CAD2020/t H2/km,Selected linear variable operating-cost model ...,Transition Accelerator H2 pipeline cost model,None,None,None,None,None,GEO001
1,R1-R0,1,H2_PIPE,1,1094.833766,CAD2020/t H2/km,Selected linear variable operating-cost model ...,Transition Accelerator H2 pipeline cost model,None,None,None,None,None,GEO001
2,R1-R2,1,H2_PIPE,1,1097.408799,CAD2020/t H2/km,Selected linear variable operating-cost model ...,Transition Accelerator H2 pipeline cost model,None,None,None,None,None,GEO001
3,R10-R11,1,H2_PIPE,1,1094.654916,CAD2020/t H2/km,Selected linear variable operating-cost model ...,Transition Accelerator H2 pipeline cost model,None,None,None,None,None,GEO001
4,R10-R17,1,H2_PIPE,1,1097.411435,CAD2020/t H2/km,Selected linear variable operating-cost model ...,Transition Accelerator H2 pipeline cost model,None,None,None,None,None,GEO001


In [12]:
# =============================================================================
# Validate all mapped H2 pipeline fixed and variable OPEX rows
# =============================================================================

PIPELINE_OPEX_VALIDATION_COLUMNS = [
    "region",
    "cost_type",
    "distance_km",
    "coefficient_per_km",
    "intercept_cost",
    "expected_cost",
    "mapped_cost",
    "difference",
]


def validate_mapped_h2_pipeline_opex(
    graph_edges: pd.DataFrame,
    opex_template: pd.DataFrame,
    mapped_costfixed: pd.DataFrame,
    mapped_costvariable: pd.DataFrame,
    tolerance: float = 1e-9,
) -> pd.DataFrame:
    """Validate every mapped H2 pipeline fixed and variable OPEX row.

    The inherited Geospatial-CANOE mapping convention is:

        cost = intercept_cost
               + coefficient_per_km * distance_km

    The fixed_opex model is validated against CostFixed, and the
    variable_opex model is validated against CostVariable.
    """

    required_graph_columns = [
        "edge_region",
        "distance_km",
    ]

    required_template_columns = [
        "cost_type",
        "coefficient_per_km",
        "intercept_cost",
    ]

    required_cost_columns = [
        "region",
        "cost",
    ]

    missing_graph_columns = [
        column
        for column in required_graph_columns
        if column not in graph_edges.columns
    ]

    missing_template_columns = [
        column
        for column in required_template_columns
        if column not in opex_template.columns
    ]

    missing_fixed_columns = [
        column
        for column in required_cost_columns
        if column not in mapped_costfixed.columns
    ]

    missing_variable_columns = [
        column
        for column in required_cost_columns
        if column not in mapped_costvariable.columns
    ]

    if missing_graph_columns:
        raise ValueError(
            "Graph-edge table is missing required columns: "
            f"{missing_graph_columns}"
        )

    if missing_template_columns:
        raise ValueError(
            "Pipeline OPEX template is missing required columns: "
            f"{missing_template_columns}"
        )

    if missing_fixed_columns:
        raise ValueError(
            "Mapped CostFixed table is missing required columns: "
            f"{missing_fixed_columns}"
        )

    if missing_variable_columns:
        raise ValueError(
            "Mapped CostVariable table is missing required columns: "
            f"{missing_variable_columns}"
        )

    expected_cost_types = {
        "fixed_opex",
        "variable_opex",
    }

    actual_cost_types = set(
        opex_template["cost_type"]
    )

    if actual_cost_types != expected_cost_types:
        raise ValueError(
            "Expected exactly fixed_opex and variable_opex "
            f"coefficients, but found {sorted(actual_cost_types)}."
        )

    if opex_template["cost_type"].duplicated().any():
        raise ValueError(
            "Pipeline OPEX template contains duplicate cost_type rows."
        )

    edge_validation_base = (
        graph_edges[
            [
                "edge_region",
                "distance_km",
            ]
        ]
        .copy()
        .rename(
            columns={
                "edge_region": "region",
            }
        )
        .sort_values("region")
        .reset_index(drop=True)
    )

    edge_validation_base["region"] = (
        edge_validation_base["region"]
        .astype("string")
        .str.strip()
    )

    edge_validation_base["distance_km"] = pd.to_numeric(
        edge_validation_base["distance_km"],
        errors="coerce",
    )

    if edge_validation_base["region"].isna().any():
        raise ValueError(
            "Graph-edge table contains missing region values."
        )

    if (edge_validation_base["region"] == "").any():
        raise ValueError(
            "Graph-edge table contains blank region values."
        )

    if edge_validation_base["region"].duplicated().any():
        raise ValueError(
            "Graph-edge table contains duplicate edge-region values."
        )

    if edge_validation_base["distance_km"].isna().any():
        raise ValueError(
            "Graph-edge table contains invalid distance_km values."
        )

    if not np.isfinite(
        edge_validation_base["distance_km"]
    ).all():
        raise ValueError(
            "Graph-edge table contains non-finite distance_km values."
        )

    if (
        edge_validation_base["distance_km"] <= 0
    ).any():
        raise ValueError(
            "Graph-edge distances must be strictly positive."
        )

    mapped_tables = {
        "fixed_opex": mapped_costfixed,
        "variable_opex": mapped_costvariable,
    }

    validation_frames = []

    for cost_type, mapped_table in mapped_tables.items():
        model_row = opex_template.loc[
            opex_template["cost_type"] == cost_type
        ]

        if len(model_row) != 1:
            raise ValueError(
                f"Expected exactly one '{cost_type}' coefficient row."
            )

        model_row = model_row.iloc[0]

        coefficient_per_km = float(
            model_row["coefficient_per_km"]
        )

        intercept_cost = float(
            model_row["intercept_cost"]
        )

        if not np.isfinite(coefficient_per_km):
            raise ValueError(
                f"{cost_type} coefficient_per_km is non-finite."
            )

        if not np.isfinite(intercept_cost):
            raise ValueError(
                f"{cost_type} intercept_cost is non-finite."
            )

        mapped_table_validation = (
            mapped_table[
                [
                    "region",
                    "cost",
                ]
            ]
            .copy()
            .rename(
                columns={
                    "cost": "mapped_cost",
                }
            )
        )

        mapped_table_validation["region"] = (
            mapped_table_validation["region"]
            .astype("string")
            .str.strip()
        )

        mapped_table_validation["mapped_cost"] = pd.to_numeric(
            mapped_table_validation["mapped_cost"],
            errors="coerce",
        )

        if mapped_table_validation["region"].duplicated().any():
            raise ValueError(
                f"Mapped {cost_type} table contains duplicate regions."
            )

        if mapped_table_validation["mapped_cost"].isna().any():
            raise ValueError(
                f"Mapped {cost_type} table contains invalid cost values."
            )

        validation = edge_validation_base.copy()

        validation["cost_type"] = cost_type
        validation["coefficient_per_km"] = coefficient_per_km
        validation["intercept_cost"] = intercept_cost

        validation["expected_cost"] = (
            validation["intercept_cost"]
            + validation["coefficient_per_km"]
            * validation["distance_km"]
        )

        validation = validation.merge(
            mapped_table_validation,
            on="region",
            how="left",
            validate="one_to_one",
        )

        if validation["mapped_cost"].isna().any():
            missing_regions = validation.loc[
                validation["mapped_cost"].isna(),
                "region",
            ].tolist()

            raise ValueError(
                f"Mapped {cost_type} table is missing graph regions: "
                f"{missing_regions[:10]}"
            )

        extra_regions = set(
            mapped_table_validation["region"]
        ) - set(
            edge_validation_base["region"]
        )

        if extra_regions:
            raise ValueError(
                f"Mapped {cost_type} table contains regions not found "
                f"in the graph-edge table: {sorted(extra_regions)[:10]}"
            )

        validation["difference"] = (
            validation["mapped_cost"]
            - validation["expected_cost"]
        )

        validation_frames.append(validation)

    pipeline_opex_validation = (
        pd.concat(
            validation_frames,
            ignore_index=True,
        )
        [PIPELINE_OPEX_VALIDATION_COLUMNS]
        .sort_values(
            [
                "cost_type",
                "region",
            ]
        )
        .reset_index(drop=True)
    )

    maximum_absolute_difference = (
        pipeline_opex_validation["difference"]
        .abs()
        .max()
    )

    if maximum_absolute_difference > tolerance:
        failed_rows = (
            pipeline_opex_validation.loc[
                pipeline_opex_validation["difference"].abs() > tolerance
            ]
            .sort_values(
                "difference",
                key=lambda series: series.abs(),
                ascending=False,
            )
            .head(10)
        )

        raise ValueError(
            "Mapped H2 pipeline fixed and variable OPEX failed "
            "full coefficient validation. "
            f"Maximum absolute difference was "
            f"{maximum_absolute_difference:.12g}, exceeding the "
            f"tolerance of {tolerance:.12g}.\n"
            f"{failed_rows}"
        )

    return pipeline_opex_validation


h2_pipeline_opex_mapping_validation = (
    validate_mapped_h2_pipeline_opex(
        graph_edges=development_graph_edges,
        opex_template=h2_pipeline_opex_coefficient_template,
        mapped_costfixed=mapped_h2_costfixed,
        mapped_costvariable=mapped_h2_costvariable,
    )
)

print(
    "Validated all mapped H2 pipeline fixed and variable "
    "OPEX coefficients."
)
print(
    "  Rows checked:                "
    f"{len(h2_pipeline_opex_mapping_validation):,}"
)
print(
    "  Edge regions checked:        "
    f"{h2_pipeline_opex_mapping_validation['region'].nunique():,}"
)
print(
    "  Cost types checked:          "
    f"{sorted(h2_pipeline_opex_mapping_validation['cost_type'].unique())}"
)
print(
    "  Maximum absolute difference: "
    f"{h2_pipeline_opex_mapping_validation['difference'].abs().max():.12g}"
)

display(h2_pipeline_opex_mapping_validation.head(10))

Validated all mapped H2 pipeline fixed and variable OPEX coefficients.
  Rows checked:                49,340
  Edge regions checked:        24,670
  Cost types checked:          ['fixed_opex', 'variable_opex']
  Maximum absolute difference: 0


,region,cost_type,distance_km,coefficient_per_km,intercept_cost,expected_cost,mapped_cost,difference
0,R0-R1,fixed_opex,41.262779,0.144804,66041.133024,66047.108047,66047.108047,0.0
1,R1-R0,fixed_opex,41.262779,0.144804,66041.133024,66047.108047,66047.108047,0.0
2,R1-R2,fixed_opex,55.541501,0.144804,66041.133024,66049.175666,66049.175666,0.0
3,R10-R11,fixed_opex,40.271042,0.144804,66041.133024,66046.964439,66046.964439,0.0
4,R10-R17,fixed_opex,55.556121,0.144804,66041.133024,66049.177783,66049.177783,0.0
5,R10-R7,fixed_opex,55.551242,0.144804,66041.133024,66049.177076,66049.177076,0.0
6,R100-R64,fixed_opex,55.570774,0.144804,66041.133024,66049.179905,66049.179905,0.0
7,R100-R99,fixed_opex,38.905650,0.144804,66041.133024,66046.766725,66046.766725,0.0
8,R1000-R1001,fixed_opex,34.909817,0.144804,66041.133024,66046.188112,66046.188112,0.0
9,R1000-R1140,fixed_opex,55.628916,0.144804,66041.133024,66049.188324,66049.188324,0.0


In [15]:
# =============================================================================
# Export topology-free H2 pipeline cost templates for build_schema.py
# =============================================================================

H2_PIPELINE_ETLSEGMENT_TEMPLATE_PATH = (
    H2_PIPELINE_COST_DIR
    / "h2_pipeline_etlsegment_template.csv"
)

H2_PIPELINE_OPEX_COEFFICIENT_PATH = (
    H2_PIPELINE_COST_DIR
    / "h2_pipeline_opex_coefficients.csv"
)


def export_h2_pipeline_cost_templates(
    etlsegment_template: pd.DataFrame,
    opex_template: pd.DataFrame,
    etlsegment_path: Path,
    opex_path: Path,
) -> None:
    """Export topology-free H2 pipeline inputs for build_schema.py.

    The ETLSegment template stores per-kilometre CAPEX bounds.
    The OPEX template stores fixed- and variable-OPEX linear coefficients.
    Neither export contains graph regions or edge distances.
    """

    etlsegment_columns = [
        "tech_or_group",
        "segment",
        "cap_lower",
        "cap_upper",
        "cost_lower_per_km",
        "cost_upper_per_km",
        "data_id",
    ]

    opex_columns = [
        "technology",
        "commodity",
        "cost_type",
        "coefficient_per_km",
        "intercept_cost",
        "units",
        "notes",
        "data_source",
        "dq_cred",
        "dq_geog",
        "dq_struc",
        "dq_tech",
        "dq_time",
        "data_id",
    ]

    missing_etlsegment_columns = [
        column
        for column in etlsegment_columns
        if column not in etlsegment_template.columns
    ]

    missing_opex_columns = [
        column
        for column in opex_columns
        if column not in opex_template.columns
    ]

    if missing_etlsegment_columns:
        raise ValueError(
            "H2 pipeline ETLSegment template is missing required "
            f"columns: {missing_etlsegment_columns}"
        )

    if missing_opex_columns:
        raise ValueError(
            "H2 pipeline OPEX template is missing required "
            f"columns: {missing_opex_columns}"
        )

    etlsegment_export = (
        etlsegment_template[
            etlsegment_columns
        ]
        .copy()
        .sort_values(
            [
                "tech_or_group",
                "segment",
            ]
        )
        .reset_index(drop=True)
    )

    opex_export = (
        opex_template[
            opex_columns
        ]
        .copy()
        .sort_values(
            [
                "technology",
                "cost_type",
            ]
        )
        .reset_index(drop=True)
    )

    if etlsegment_export.empty:
        raise ValueError(
            "H2 pipeline ETLSegment template is empty."
        )

    if opex_export.empty:
        raise ValueError(
            "H2 pipeline OPEX coefficient template is empty."
        )

    expected_opex_cost_types = {
        "fixed_opex",
        "variable_opex",
    }

    actual_opex_cost_types = set(
        opex_export["cost_type"]
    )

    if actual_opex_cost_types != expected_opex_cost_types:
        raise ValueError(
            "H2 pipeline OPEX export must contain exactly "
            f"{sorted(expected_opex_cost_types)}, but contains "
            f"{sorted(actual_opex_cost_types)}."
        )

    if opex_export["cost_type"].duplicated().any():
        raise ValueError(
            "H2 pipeline OPEX export contains duplicate cost_type rows."
        )

    if etlsegment_export[
        [
            "tech_or_group",
            "segment",
        ]
    ].duplicated().any():
        raise ValueError(
            "H2 pipeline ETLSegment export contains duplicate "
            "technology-segment rows."
        )

    if not (
        etlsegment_export["cap_upper"]
        > etlsegment_export["cap_lower"]
    ).all():
        raise ValueError(
            "ETLSegment export contains invalid capacity bounds."
        )

    if not (
        etlsegment_export["cost_upper_per_km"]
        > etlsegment_export["cost_lower_per_km"]
    ).all():
        raise ValueError(
            "ETLSegment export contains invalid CAPEX bounds."
        )

    etlsegment_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    opex_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    etlsegment_export.to_csv(
        etlsegment_path,
        index=False,
    )

    opex_export.to_csv(
        opex_path,
        index=False,
    )

    print("Exported topology-free H2 pipeline cost templates.")
    print(
        "  ETLSegment rows: "
        f"{len(etlsegment_export):,}"
    )
    print(
        "  OPEX rows:       "
        f"{len(opex_export):,}"
    )
    print(
        "  ETLSegment:      "
        f"{etlsegment_path}"
    )
    print(
        "  OPEX:            "
        f"{opex_path}"
    )


export_h2_pipeline_cost_templates(
    etlsegment_template=h2_etlsegment_template,
    opex_template=h2_pipeline_opex_coefficient_template,
    etlsegment_path=H2_PIPELINE_ETLSEGMENT_TEMPLATE_PATH,
    opex_path=H2_PIPELINE_OPEX_COEFFICIENT_PATH,
)

Exported topology-free H2 pipeline cost templates.
  ETLSegment rows: 4
  OPEX rows:       2
  ETLSegment:      C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\processed\costs\transport\pipelines\h2_pipeline\h2_pipeline_etlsegment_template.csv
  OPEX:            C:\Users\Andrew Vigars\Research\repos\temoa-upstream\temoa_geospace\data_files\processed\costs\transport\pipelines\h2_pipeline\h2_pipeline_opex_coefficients.csv
